# 초구 예측 — model_10 (채택X·보류: 전체 투구 모델 블렌딩)

- 0장 = 모델 전 확인
  - 0-1 리그 최근 잔차: 2019 신호 없음, 적용 안 함 (README 4-20)
- 모델: model_09 구종 확률에 **전체 투구 모델(all_pitch_01, 수준 2) 0-0 확률을 섞음** P = 0.8·P_model_09 + 0.2·P_전체 (사용자 결정 2026-11-10)
  - 전체 투구 모델과 섞는 비율은 `model/all_pitch_01.ipynb`에서 2019 교차로 고름 (README 4-21): 수준 2 반감 50·k1 10·k2 10, 고정 w 0.2
  - model_09 부분(B4·LightGBM 월 단위 재학습)과 위치는 그대로 -> 바꾸는 것은 블렌딩 하나
- 비교 대상: model_09 -> **적용 안 함 (보류)**: log loss는 좋아지지만 1등 정확도·macro F1 하락, 나중에 그대로 가져옴 (사용자 결정 2026-11-10)

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc

MODEL = "model_10"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)

초구 50,709개 (검증 13,458 / 평가 12,615)


## 0. 모델 전 확인

### 0-1. 리그 최근 잔차: 어긋남을 미리 알 수 있는가

- 배경 (model_09 0-2): 직구 비율이 한 달 단위로 몇 %p 어긋나는 달이 있음 (2019 6월 −1.8%p, 2020 6월 −2.7%p·7월 +3.2%p, 예측 − 실제 반대 부호 주의), model_07과 B4가 같이 어긋남
- 어긋난 달을 보고 고치면 답을 아는 셈 -> **전날까지 정보만으로** 앞으로의 어긋남을 예측할 수 있는지 확인
- 세 가지 가능성: ① 우연(표본 잡음) -> 못 잡음 / ② 리그 전체가 한동안 한쪽으로 기움 -> 최근 리그 잔차로 잡을 수 있음 / ③ 특정 투수 배합 변화 -> B4가 이미 따라감
- 방법 (새 학습 없음, model_09 결과 사용)
  - 공별 잔차 = 실제(직구면 1) − model_09 직구 확률, 직구 계열(직구·투심·커터·싱커)도 같이
  - 각 공에 **전날까지 최근 N일(7·14·28일, 같은 시즌 안) 리그 잔차 평균**을 붙임 -> 그 공 잔차를 이 값으로 회귀 (기울기 β, 경기 단위 부트스트랩)
    - β ≈ 0: 앞 기간 어긋남이 다음 공을 예측 못 함 = ① / β > 0이고 구간이 0 위: ② -> 변수로 넣을 후보
  - 같은 내용을 겹치지 않는 N일 구간으로도: 앞 구간 잔차 vs 다음 구간 잔차 상관
  - 판단은 2019, 2020은 확인만

In [2]:
P13 = pd.read_parquet("results/model_09/per_pitch.parquet")
FAM = ["FAST", "TWOS", "CUTT", "SINK"]  # 직구 계열
TARGETS = {"직구": ["FAST"], "직구 계열": FAM}
WINDOWS = [7, 14, 28]

def league_resid(yr, msk):  # 그 해 공별: 날짜, 경기, 잔차(대상별)
    d = fc.align(P13, fp, yr)
    out = pd.DataFrame({"date": pd.to_datetime(d["gameID"].str[:8]).to_numpy(), "g": d["gameID"].to_numpy()})
    for nm, cs in TARGETS.items():
        out[nm] = d["pitch_type"].isin(cs).astype(float).to_numpy() - d[["p_type_" + c for c in cs]].sum(axis=1).to_numpy()
    return out

def past_mean(r, col, n):  # 각 공에 전날까지 최근 n일 리그 잔차 평균 (공 수 가중, 같은 시즌, 없으면 NaN)
    daily = r.groupby("date")[col].agg(["sum", "count"])
    days = daily.index.to_numpy()
    cs, cc = np.r_[0, daily["sum"].cumsum().to_numpy()], np.r_[0, daily["count"].cumsum().to_numpy()]
    hi = np.searchsorted(days, r["date"].to_numpy(), "left")  # 그날 전까지
    lo = np.searchsorted(days, r["date"].to_numpy() - np.timedelta64(n, "D"), "left")
    s, c = cs[hi] - cs[lo], cc[hi] - cc[lo]
    return np.where(c >= 200, s / np.maximum(c, 1), np.nan), c  # 공 200개 미만이면 빈 값 (시즌 첫날들)

def slope(x, y):  # y = a + β x
    x = x - x.mean()
    return (x * (y - y.mean())).sum() / (x * x).sum()

def slope_ci(x, y, g, B=1000):  # 경기 단위 부트스트랩
    df = pd.DataFrame({"g": g, "x": x, "y": y, "xx": x * x, "xy": x * y})
    gs = df.groupby("g").agg(sx=("x", "sum"), sy=("y", "sum"), sxx=("xx", "sum"), sxy=("xy", "sum"), n=("x", "size"))
    A = gs[["sx", "sy", "sxx", "sxy", "n"]].to_numpy()
    idx = np.random.default_rng(0).integers(0, len(A), (B, len(A)))
    T = A[idx].sum(1)  # (B, 5)
    sx, sy, sxx, sxy, n = T.T
    return np.percentile((sxy - sx * sy / n) / (sxx - sx * sx / n), [2.5, 97.5])

rows = []
R = {}
for yr, msk in ((fc.VAL_YEAR, val), (fc.TEST_YEAR, test)):
    r = league_resid(yr, msk)
    R[yr] = r
    for nm in TARGETS:
        for n in WINDOWS:
            x, cnt = past_mean(r, nm, n)
            k = ~np.isnan(x)
            lo, hi = slope_ci(x[k], r[nm].to_numpy()[k], r["g"].to_numpy()[k])
            rows.append({"연도": yr, "대상": nm, "최근 N일": n, "쓴 공": int(k.sum()), "최근 잔차 SD (%p)": np.nanstd(x) * 100,
                         "기울기 β": slope(x[k], r[nm].to_numpy()[k]), "95% 하한": lo, "95% 상한": hi})
display(pd.DataFrame(rows).set_index(["연도", "대상", "최근 N일"]).round(3))

쓴 공  최근 잔차 SD (%p)  기울기 β  95% 하한  95% 상한
연도   대상    최근 N일                                             
2019 직구    7      12976          2.408 -0.110  -0.492   0.262
           14     13237          1.672 -0.116  -0.676   0.432
           28     13237          1.281 -0.017  -0.769   0.687
     직구 계열 7      12976          2.534 -0.191  -0.563   0.176
           14     13237          1.731  0.064  -0.514   0.611
           28     13237          1.322  0.055  -0.644   0.705
2020 직구    7      12343          3.204  0.359   0.052   0.632
           14     12343          2.587  0.262  -0.125   0.630
           28     12343          1.929  0.187  -0.377   0.760
     직구 계열 7      12343          3.383  0.279  -0.021   0.548
           14     12343          2.462  0.021  -0.362   0.374
           28     12343          1.529 -0.303  -0.990   0.342

In [3]:
# 겹치지 않는 N일 구간: 앞 구간 리그 잔차 vs 바로 다음 구간 리그 잔차 (구간마다 잔차 평균, 공 수 가중)
rows = []
for yr, r in R.items():
    day0 = r["date"].min()
    for n in WINDOWS:
        blk = ((r["date"] - day0).dt.days // n).to_numpy()
        for nm in TARGETS:
            b = r.assign(blk=blk).groupby("blk")[nm].agg(["mean", "count"])
            b = b[b["count"] >= 200]
            a, c = b["mean"].to_numpy()[:-1], b["mean"].to_numpy()[1:]
            ok = np.diff(b.index.to_numpy()) == 1  # 바로 다음 구간만
            rows.append({"연도": yr, "대상": nm, "N일": n, "구간 수": len(b), "구간 잔차 SD (%p)": b["mean"].std() * 100,
                         "잡음만일 때 SD (%p)": np.sqrt((0.25 / b["count"]).mean()) * 100,
                         "앞-다음 상관": np.corrcoef(a[ok], c[ok])[0, 1] if ok.sum() > 2 else np.nan})
display(pd.DataFrame(rows).set_index(["연도", "대상", "N일"]).round(3))

구간 수  구간 잔차 SD (%p)  잡음만일 때 SD (%p)  앞-다음 상관
연도   대상    N일                                              
2019 직구    7     27          2.705           2.290   -0.215
     직구 계열 7     27          2.876           2.290   -0.134
     직구    14    14          1.661           1.660    0.182
     직구 계열 14    14          2.000           1.660    0.370
     직구    28     7          1.256           1.151    0.425
     직구 계열 28     7          1.828           1.151    0.092
2020 직구    7     26          3.398           2.328    0.233
     직구 계열 7     26          3.486           2.328    0.097
     직구    14    13          2.589           1.642    0.121
     직구 계열 14    13          2.575           1.642   -0.261
     직구    28     7          2.277           1.327   -0.261
     직구 계열 28     7          2.010           1.327   -0.494

#### 🔍 0-1 결과
#### 2019: 앞 기간 리그 잔차로 다음 어긋남을 예측 못 함 = 대부분 표본 잡음 -> 적용 안 함
- 기울기 β (전날까지 최근 N일 리그 잔차 -> 그 공 잔차, 1이면 앞 어긋남이 그대로 이어짐, 0이면 무관)
  - 2019 (판단) 직구: 7일 −0.11 · 14일 −0.12 · 28일 −0.02, 직구 계열: −0.19 · +0.06 · +0.06 -> 모두 95% 구간에 0 포함, 부호도 섞임
  - 2020 (확인) 직구: 7일 +0.36 (95% +0.05 ~ +0.63) · 14일 +0.26 · 28일 +0.19 (둘 다 0 포함), 직구 계열 7일 +0.28 (0 포함) -> 짧은 구간에서 약한 이어짐
- 겹치지 않는 구간의 잔차 SD vs 잡음만일 때 SD (경기 안 공끼리 비슷한 몫을 빼서 실제 잡음은 이보다 큼)
  - 2019 직구: 7일 2.7 vs 2.3, 14일 1.7 vs 1.7, 28일 1.3 vs 1.2 (%p) -> 잡음과 거의 같음 = 2019 어긋남(6월 등)은 ① 우연
  - 2020 직구: 3.4 vs 2.3, 2.6 vs 1.6, 2.3 vs 1.3 -> 잡음보다 큼 = 2020에는 실제 리그 흔들림이 있었음 (늦은 개막·빽빽한 일정의 특이 시즌일 수 있음)
  - 앞-다음 구간 상관은 두 해 모두 −0.5 ~ +0.4로 섞임 (구간 7–27개라 불안정)
- 정리: 2019에서는 잡을 신호가 없음 -> 규칙상 적용 안 함 (2020의 약한 7일 이어짐을 보고 넣으면 2020으로 고르는 셈)
  - 2020 흔들림은 크기가 최대 3%p 정도, 방향도 6월 -> 7월 뒤집힘 -> 최근 잔차를 넣어도 7월은 더 틀렸을 것
- 남는 메모: 다른 시즌 데이터가 생기면(2021 이후) 같은 표로 다시 확인 -> 2020처럼 실제 흔들림이 있는 시즌이 흔하면 "리그 최근 잔차" 변수 후보


## 모델

In [4]:
# 0장이 모델 변수 이름을 덮어쓰지 않았는지 대비: 설정 상수를 다시 정함
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]


## 1. 파라미터

- model_09 값 그대로 (구종 B4, 변수 `FEAT`, LightGBM 잎 7·잎당 최소 20·λ 1000·트리 730·월 단위 재학습, 위치는 model_03)
- 바꾸는 것: 블렌딩 `BLEND` (all_pitch_01 결과 `results/all_pitch_01/p00.parquet`·`blend_params.csv`를 읽음)

In [5]:
# ① 구종 출발점 B4: model_06 값 (고정) - 반감기·alpha·hand_k, 시즌 초 배수 (기준점 = 지난 시즌 전부)
B4_P = dict(half_life=35.4, alpha=7.5, hand_k=5.48, season_f=0.25)
# ① LightGBM 변수: A 이번 타석 상황 + 앞 타석 초구 결과 원-핫 (fc.FIRST_CATS)
FEAT = fc.A_COLS + ["prev_first_" + c for c in fc.FIRST_CATS]
# ② 위치 (model_03, 고정)
LOC_REF, LOC_REF_HL = "decay", 173.0
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 17.3
ZONE_M = 32
# ① LightGBM: model_07 값 (고정)
LGB_P = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}
LGB_ROUNDS = 730
RETRAIN = "month"  # 재학습 주기: 그 달 첫 경기 전날까지 데이터로 다시 학습 (3월은 4월에, 10월은 9월에 합침)
# ① 블렌딩: 전체 투구 모델 (all_pitch_01에서 고름)
BLEND = pd.read_csv("results/all_pitch_01/blend_params.csv", index_col=0).iloc[:, 0].to_dict()
assert BLEND["level"] == 2 and BLEND["half_life"] == 50 and BLEND["w"] == 0.2, BLEND
print("블렌딩:", BLEND)
# 구종 보정 학습 해
TRAIN_FIRST = 2017

블렌딩: {'level': 2.0, 'half_life': 50.0, 'season_f': 0.25, 'k1': 10.0, 'k2': 10.0, 'w': 0.2}


## 2. ① 구종: model_09 (B4 + LightGBM 월 단위 재학습) + 전체 투구 모델 블렌딩

### 2-1. B4 (model_06–09과 같음)

In [6]:
def b4_online(q, year, half_life, alpha, hand_k, season_f):  # model_06 B4 (n, 9): 지난 시즌 리그 비율 -> 투수 (alpha) -> 투수×손 (hand_k), 시즌 초 배수
    first = year == fp["GYEAR"].min()  # 첫 시즌(2017): 지난 기록 없음 -> 그 시즌 비율 (학습 데이터 출발점용)
    ref = fp.loc[fp["GYEAR"] == year if first else fp["GYEAR"] < year, CLASSES].mean().to_numpy()
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    S, N, S2, N2 = out
    return fc.shrink(S2, N2, fc.shrink(S, N, ref, alpha), hand_k)

K = len(CLASSES)
P_B4 = np.full((len(fp), K), np.nan)  # 2017–2020
for yr in range(TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P_B4[k] = b4_online(fp[k], yr, **B4_P)
b4_19 = fc.prob_metrics(P_B4[val], y_type[val])["logloss"]
assert abs(b4_19 - 1.09367) < 1e-5, b4_19  # model_06–09 B4의 2019 log loss
print(f"2019 B4 log loss {b4_19:.5f} = model_06–09 B4 (재현 확인)")
m09 = pd.read_parquet("results/model_09/per_pitch.parquet")  # 비교 대상

2019 B4 log loss 1.09367 = model_06–09 B4 (재현 확인)


### 2-2. 변수 (model_07·09과 같음)

- A 이번 타석 상황 + 앞 타석 초구 결과, 미래 정보 검사 다시 통과 확인

In [7]:
X_all = pd.concat([fc.situation_features(m, fp), fc.game_flow_features(m, fp)], axis=1)[FEAT]
assert X_all.notna().all().all()
for fn in (fc.situation_features, fc.game_flow_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(TRAIN_FIRST)], n=150)
    assert bad.empty, bad
display(X_all.describe().T[["mean", "min", "max"]].round(3))

미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개


,mean,min,max
pc,33.394,0.0,124.0
tto,1.592,1.0,5.0
starter,0.690,0.0,1.0
inning,4.548,1.0,12.0
out_cn,0.970,0.0,2.0
base1,0.359,0.0,1.0
base2,0.194,0.0,1.0
base3,0.100,0.0,1.0
batorder,4.811,1.0,9.0
sdiff,0.369,-21.0,19.0


### 2-3. LightGBM 월 단위 재학습 (model_09과 같음)

- 달마다 그 달 첫 경기 전날까지 모든 초구로 다시 학습 -> 확률이 model_09과 같아야 함 (재현 확인)

In [8]:
y_all = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
INIT = np.log(np.clip(P_B4, 1e-6, None))
INIT -= INIT.mean(1, keepdims=True)  # 출발점 (공마다 상수는 softmax에 영향 없음)
BASE = dict(objective="multiclass", num_class=K, learning_rate=0.03, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
            verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)  # model_05–07과 같음
yr_all = fp["GYEAR"].to_numpy()
DATE = pd.to_datetime(fp["gameID"].str[:8]).to_numpy()
MON = np.clip(fp["gameID"].str[4:6].astype(int).to_numpy(), 4, 9)  # 재학습 묶음 달: 3월 -> 4월, 10월 -> 9월

def fit_rows(tr):  # 학습 행 tr로 LightGBM (트리 수 고정)
    return lgb.train({**BASE, **LGB_P}, lgb.Dataset(X_all.iloc[tr], y_all[tr], init_score=INIT[tr]), LGB_ROUNDS)

def type_prob(bst, idx):  # B4 출발점 + 트리 보정 -> 구종 확률 (n, 9)
    return np.exp(log_softmax(bst.predict(X_all.iloc[idx], raw_score=True) + INIT[idx], axis=1))

P_TYPE_ALL = np.full((len(fp), K), np.nan)
BST, plan = {}, []
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    iy = np.flatnonzero(yr_all == yr)
    for mon in np.unique(MON[iy]):
        te = iy[MON[iy] == mon]
        start = DATE[te].min()  # 그 달 첫 경기일
        tr = np.flatnonzero((yr_all >= TRAIN_FIRST) & (DATE < start))
        assert DATE[tr].max() < start  # 미래 정보 없음
        BST[(yr, mon)] = bst = fit_rows(tr)
        P_TYPE_ALL[te] = type_prob(bst, te)
        plan.append({"연도": yr, "월": mon, "예측 초구": len(te), "학습 초구": len(tr), "그 시즌 학습 초구": int((yr_all[tr] == yr).sum()),
                     "학습 끝": str(pd.Timestamp(DATE[tr].max()).date()), "예측 시작": str(pd.Timestamp(start).date())})
display(pd.DataFrame(plan).set_index(["연도", "월"]))
P13 = {yr: fc.align(m09, fp, yr)[fc.P_TYPE].to_numpy() for yr in (fc.VAL_YEAR, fc.TEST_YEAR)}  # model_09 구종 확률 (fp 그 해 순서)
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    assert np.allclose(P_TYPE_ALL[msk], P13[yr], atol=1e-6)
print("월 단위 재학습 구종 확률 = model_09 (재현 확인)")

예측 초구  학습 초구  그 시즌 학습 초구        학습 끝       예측 시작
연도   월                                                  
2019 4   2568  24636           0  2018-10-14  2019-03-23
     5   2403  27204        2568  2019-04-30  2019-05-01
     6   2493  29607        4971  2019-05-31  2019-06-01
     7   1791  32100        7464  2019-06-30  2019-07-02
     8   2362  33891        9255  2019-07-31  2019-08-01
     9   1841  36253       11617  2019-08-31  2019-09-01
2020 5   2264  38094           0  2019-10-01  2020-05-05
     6   2379  40358        2264  2020-05-31  2020-06-02
     7   2233  42737        4643  2020-06-30  2020-07-01
     8   1985  44970        6876  2020-07-31  2020-08-01
     9   3754  46955        8861  2020-08-30  2020-09-01

월 단위 재학습 구종 확률 = model_09 (재현 확인)


### 2-4. 전체 투구 모델 블렌딩

- 전체 투구 모델 0-0 확률 (all_pitch_01 수준 2): 투수의 모든 카운트 공에서 "그 카운트·타자 손 리그 배합 대비 비율"을 전날까지 누적 -> 0-0 리그 배합에 곱함
- P_구종 = (1 − w)·P_model_09 + w·P_전체, w 0.2 (고정)
- 확인: 2019·2020 구종 log loss·top1, B4 유효 초구 수별 이득, 구종 1등 보정 구간표 (model_09 vs model_10)

In [9]:
AP = pd.read_parquet("results/all_pitch_01/p00.parquet")
P_AP = {yr: fc.align(AP, fp, yr)[fc.P_TYPE].to_numpy() for yr in (fc.VAL_YEAR, fc.TEST_YEAR)}  # 전체 투구 모델 (fp 그 해 순서)
NB = {yr: fc.align(AP, fp, yr)["b4_n"].to_numpy() for yr in (fc.VAL_YEAR, fc.TEST_YEAR)}  # B4 투수 유효 초구 수
assert all(not np.isnan(P_AP[yr]).any() for yr in P_AP)
P_BLEND = P_TYPE_ALL.copy()
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    P_BLEND[msk] = (1 - BLEND["w"]) * P_TYPE_ALL[msk] + BLEND["w"] * P_AP[yr]
rows = {}
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    for nm, P in [("B4", P_B4[msk]), ("전체 투구 모델만", P_AP[yr]), ("model_09", P13[yr]), (MODEL, P_BLEND[msk])]:
        r = fc.prob_metrics(P, y_type[msk])
        rows[(yr, nm)] = {"log loss": r["logloss"], "top1": r["top1"], "top2": r["top2"]}
display(pd.DataFrame(rows).T.round(4))
nll = lambda P, y: -np.log(np.clip(P[np.arange(len(y)), y], 1e-12, None))
rows = []
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    d = nll(P13[yr], y_type[msk]) - nll(P_BLEND[msk], y_type[msk])  # 양수 = 블렌딩이 좋음
    b = pd.cut(NB[yr], [-0.1, 5, 20, 40, 100], labels=["0–5", "5–20", "20–40", "40+"])
    for lab, g in pd.Series(d).groupby(b.to_numpy(), observed=True):
        rows.append({"연도": yr, "B4 유효 초구 수": lab, "공": len(g), "이득 ×1000": g.mean() * 1000})
display(pd.DataFrame(rows).set_index(["B4 유효 초구 수", "연도"])["이득 ×1000"].unstack().round(2))
# 구종 1등 보정 구간표 (2019·2020): model_09 vs model_10
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    for nm, P in [("model_09", P13[yr]), (MODEL, P_BLEND[msk])]:
        rep, tabs = fc.calib_report(P, y_type[msk], CLASSES)
        t = tabs["1등 예측"]
        print(f"{yr} {nm}: 1등 ECE {rep.loc['1등 예측', 'ECE']:.4f}")
        display(t[t["공수"] >= 30].round(3).T)

log loss    top1    top2
2019 B4           1.0937  0.5279  0.8159
     전체 투구 모델만    1.0970  0.5279  0.8123
     model_09     1.0613  0.5385  0.8177
     model_10     1.0566  0.5360  0.8201
2020 B4           1.0916  0.5264  0.8179
     전체 투구 모델만    1.1043  0.5214  0.8171
     model_09     1.0561  0.5332  0.8218
     model_10     1.0524  0.5336  0.8242

연도,2019,2020
B4 유효 초구 수,,
0–5,16.33,NaN
20–40,10.49,6.49
40+,3.36,3.11
5–20,9.12,3.46


2019 model_09: 1등 ECE 0.0128


,20%~30%,30%~40%,40%~50%,50%~60%,60%~70%,70%~80%,80%~90%,90%~100%
공수,92.000,1853.000,3886.000,3442.000,2267.000,1219.000,560.000,139.000
예측평균,0.284,0.362,0.453,0.546,0.646,0.744,0.846,0.925
실제비율,0.304,0.378,0.452,0.525,0.628,0.742,0.871,0.986
차이,0.020,0.017,-0.000,-0.021,-0.018,-0.002,0.026,0.061


2019 model_10: 1등 ECE 0.0112


,20%~30%,30%~40%,40%~50%,50%~60%,60%~70%,70%~80%,80%~90%,90%~100%
공수,58.000,1753.000,4111.000,3716.000,2312.000,1089.000,382.000,37.000
예측평균,0.285,0.364,0.452,0.547,0.645,0.741,0.837,0.926
실제비율,0.259,0.356,0.446,0.542,0.642,0.794,0.903,0.946
차이,-0.027,-0.008,-0.006,-0.005,-0.003,0.053,0.066,0.020


2020 model_09: 1등 ECE 0.0155


,20%~30%,30%~40%,40%~50%,50%~60%,60%~70%,70%~80%,80%~90%,90%~100%
공수,231.000,1815.000,3452.000,3141.000,1925.000,1225.000,688.000,138.000
예측평균,0.271,0.363,0.452,0.546,0.644,0.744,0.844,0.924
실제비율,0.277,0.364,0.434,0.525,0.626,0.746,0.874,0.978
차이,0.006,0.000,-0.018,-0.021,-0.019,0.002,0.030,0.054


2020 model_10: 1등 ECE 0.0178


,20%~30%,30%~40%,40%~50%,50%~60%,60%~70%,70%~80%,80%~90%
공수,233.000,1696.000,3613.000,3306.000,2067.000,1254.000,423.000
예측평균,0.265,0.364,0.452,0.546,0.646,0.745,0.835
실제비율,0.240,0.356,0.441,0.525,0.648,0.795,0.905
차이,-0.025,-0.009,-0.010,-0.021,0.002,0.050,0.071


#### 🔍 블렌딩 진단
#### 구종 log loss 2019 −0.0047, 2020 −0.0037, 기록 적은 투수에서 크게 좋아짐 / 대신 1등이 직구 쪽으로 몰려 macro F1이 떨어짐
- 구종 log loss: 2019 1.0613 -> 1.0566 (all_pitch_01 정직한 교차 +4.2보다 큼 = w를 2019 전체로 고른 몫), 2020 1.0561 -> 1.0524 (−0.0037)
- B4 유효 초구 수별 이득 ×1000: 0–5개 +16 (2019), 5–20개 +9.1 · +3.5, 20–40개 +10.5 · +6.5, 40개+ +3.4 · +3.1 -> 모든 구간 좋아짐, 기록 적을수록 큼
- 전체 투구 모델은 직구 쪽 확률이 큼 -> 섞으면 1등이 직구로 바뀌는 공이 늘어남 (1등 직구 2019 10,267 -> 10,717, 2020 8,756 -> 9,088)
  - 직구 recall 0.869 -> 0.892, 대신 커브 0.078 -> 0.039·포크 0.222 -> 0.149·체인지업 0.287 -> 0.256 (2019) -> macro F1 2019 0.323 -> 0.292, 2020 0.355 -> 0.341
  - top1은 2019 −0.25%p, 2020 +0.04%p / top2·top3는 두 해 모두 좋아짐 (+0.24%p, +0.1–0.2%p)
- 구종 1등 ECE: 2019 0.0128 -> 0.0112, 2020 0.0155 -> 0.0178 -> 2020은 70–90% 구간 과소 확신(실제가 예측보다 5–7%p 높음)이 커짐


## 3. ② 위치 (model_03과 같음)

In [11]:
# 24칸 적분 함수는 fp_common으로 옮김: fc.zone_from_gauss_cov(g, q, m=32) (계산 설명은 위, 코드 주석은 fp_common.py)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

In [12]:
L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 리그 구종×타자 손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 리그 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 리그 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None):  # 위치 모멘트 + 24칸 비율 (n, 29): L2
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    S1, N1 = fc.lookup(fc.prior_table(fp, K1, L_COLS, half_life), q, K1, L_COLS)
    P = fc.shrink(S1, N1, P, k_type)  # 투수×구종 쪽으로 k_type만큼
    S2, N2 = fc.lookup(fc.prior_table(fp, K2, L_COLS, half_life), q, K2, L_COLS)
    return fc.shrink(S2, N2, P, k_hand)  # 투수×구종×손 쪽으로 k_hand만큼

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, cov=True):  # 구종이 t라고 가정했을 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)  # 구종만 t로 바꾼 사본
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl), qt)  # 모멘트 -> 정규분포 (높이는 그 타자 존으로 cm 환산)
    return zone_probs(g, qt, cov), g  # 정규분포를 24칸 위에서 적분 (cov=True 공분산 반영)

## 4. 실전 예측 (예측 구종으로 위치 이어서)

- model_09과 같음, 구종 확률만 블렌딩 결과(`P_BLEND`)

In [13]:
def predict(q, year, Pt):  # 초구 q의 공별 예측·점수
    n, i = len(q), np.arange(len(q))
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()  # 실제 구종·칸 번호
    nll = lambda p: -np.log(np.clip(p, 1e-12, None))  # 공별 log loss (prob_metrics와 같은 방식으로 자름)

    # ① 구종 확률 (n, 9): 2장 결과를 받음

    # ② 위치: 구종 9개 각각 가정 -> 24칸 확률, 실제 위치의 로그 밀도, 예측 중심 (좌우, 높이)
    PZ, LD, MX, MZ = [], [], [], []
    for t in CLASSES:
        P, g = loc_given_type(q, year, t, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL)
        PZ.append(P); LD.append(fc.log_dens(g, q)); MX.append(g[0]); MZ.append(g[1])
    PZ, LD, MX, MZ = np.stack(PZ, 1), np.stack(LD, 1), np.stack(MX, 1), np.stack(MZ, 1)  # (n, 구종, ...)
    joint = Pt[:, :, None] * PZ  # 구종 확률 × 그 구종일 때 칸 확률 = 구종×칸 결합 확률 (n, 9, 24)
    P_zone = joint.sum(1)  # 실전 24칸 확률 = 구종으로 합침
    # 좌표: 구종 확률로 섞은 정규분포에서 실제 위치의 로그 밀도
    ld_mix = np.log(np.einsum("nt,nt->n", Pt, np.exp(LD - LD.max(1, keepdims=True)))) + LD.max(1)  #  공마다 최댓값 빼기 -> exp -> 섞기 -> log (계산 안전)
    dist_true = np.hypot(q["x"] - MX[i, yt], q["CROSS_PLATE_Z_CM"] - MZ[i, yt])  # 실제 구종 중심과의 거리 cm
    dist_mix = np.hypot(q["x"] - (Pt * MX).sum(1), q["CROSS_PLATE_Z_CM"] - (Pt * MZ).sum(1))  # 구종 확률로 가중한 중심과의 거리 cm

    t_hat = np.array(CLASSES)[Pt.argmax(1)]  # 1등 구종 이름

    # 공별 결과 표 (필수: 키, 구종 확률 9개, 실전 24칸 확률 / 나머지는 선택 열)
    return pd.concat([
        q[["gameID", "pit_no", "pitcher_code", "GYEAR", "pitch_type", "zone"]].reset_index(drop=True),
        pd.DataFrame(Pt, columns=fc.P_TYPE),  # p_type_<구종>
        pd.DataFrame(P_zone, columns=fc.P_ZONE),  # p_zone_<칸>
        pd.DataFrame({"t_hat": t_hat}),  # 1등 구종 이름
        pd.DataFrame({
            "ll_type": nll(Pt[i, yt]),  # 구종 log loss
            "ll_zone": nll(P_zone[i, yz]),  # 24칸 log loss (실전)
            "ll_zone_true": nll(PZ[i, yt, yz]),  # 24칸 log loss (실제 구종)
            "ll_joint": nll(joint[i, yt, yz]),  # 구종×칸 log loss
            "nll_coord": -ld_mix,  # 좌표 NLL (실전)
            "nll_coord_true": -LD[i, yt],  # 좌표 NLL (실제 구종)
            "dist": np.asarray(dist_mix),  # 좌표 거리 cm (실전)
            "dist_true": np.asarray(dist_true),  # 좌표 거리 cm (실제 구종)
        })], axis=1)

In [14]:
# 2019 검증 + 2020 평가 초구 전부 예측
pp = pd.concat([predict(q_val, fc.VAL_YEAR, P_BLEND[val]), predict(q_test, fc.TEST_YEAR, P_BLEND[test])], ignore_index=True)
print(pp.shape)
# 확인: 구종과 무관한 열(실제 구종 기준 위치)은 model_09과 같아야 함
chk = pp.merge(m09, on=fc.KEYS, suffixes=("", "_03"))
same = ["ll_zone_true", "nll_coord_true", "dist_true"]
bad = [c for c in same if not np.allclose(chk[c], chk[c + "_03"], equal_nan=True)]
assert len(chk) == len(pp) and not bad, f"model_09과 다른 열: {bad}"
print("실제 구종 기준 위치 = model_09 (재현 확인)")

(26073, 48)
실제 구종 기준 위치 = model_09 (재현 확인)


## 5. 평가

- 채점은 `fc.evaluate`, model_09과 나란히 / 경기 단위 부트스트랩 (model_10 − model_09, 2020)
- 2020은 확인용 -> 2020을 보고 파라미터 다시 고르지 않음

In [15]:
def eval_years(pp_):  # 2019·2020 주요 지표 표
    return pd.DataFrame({"검증 2019": fc.evaluate(pp_, fp, fc.VAL_YEAR), "평가 2020": fc.evaluate(pp_, fp, fc.TEST_YEAR)})

summary = eval_years(pp)
m09_sum = eval_years(m09)
display(pd.concat({MODEL: summary, "model_09": m09_sum, "차이 (model_09 대비)": summary - m09_sum}, axis=1).round(4))

model_10          model_09          차이 (model_09 대비)        
                     검증 2019  평가 2020  검증 2019  평가 2020          검증 2019 평가 2020
구종 logloss            1.0566   1.0524   1.0613   1.0561          -0.0047 -0.0037
구종 top1               0.5360   0.5336   0.5385   0.5332          -0.0025  0.0004
구종 top2               0.8201   0.8242   0.8177   0.8218           0.0024  0.0024
구종 top3               0.9385   0.9433   0.9374   0.9412           0.0010  0.0021
구종 macro F1           0.2920   0.3411   0.3226   0.3545          -0.0306 -0.0135
24칸 logloss (실전)      2.9493   2.9454   2.9493   2.9454           0.0000 -0.0001
24칸 top1              0.1297   0.1337   0.1300   0.1337          -0.0003  0.0001
24칸 top3              0.2896   0.2920   0.2888   0.2908           0.0007  0.0012
H/S/C logloss         1.0748   1.0768   1.0748   1.0767           0.0001  0.0000
H/S/C 정확도             0.3963   0.3900   0.3951   0.3896           0.0012  0.0004
24칸 logloss (실제 구종)   2.8933   2.8967   2.8933   2.8967           0.0000  0.0000
구종×칸 logloss          3.9498   3.9484   3.9546   3.9521          -0.0047 -0.0037
좌표 평균거리 cm           32.6878  31.8341  32.6837  31.8305           0.0041  0.0036
좌표 NLL                9.3171   9.2660   9.3171   9.2660          -0.0000 -0.0000

In [16]:
display(fc.bootstrap_diff(fc.pitch_scores(m09, fp), fc.pitch_scores(pp, fp)).round(4))  # 음수 = 이 모델이 좋음 (b − a)
# 실전 출력: 누적 80% 보여 주기 (2020)
for nm, P_ in [("model_09", m09), (MODEL, pp)]:
    d = fc.align(P_, fp, fc.TEST_YEAR)
    Pt_, yt_ = d[fc.P_TYPE].to_numpy(), d["pitch_type"].map(fc.TYPE_IDX).to_numpy()
    s = fc.show_types(Pt_)
    print(f"{nm}: 보여 준 구종 수 평균 {s.sum(1).mean():.2f}, 포함률 {s[np.arange(len(yt_)), yt_].mean():.3f}")

,차이 (b−a),95% 하한,95% 상한,판정
ll_type,-0.0037,-0.0055,-0.0018,좋아짐
ll_zone,-0.0001,-0.0004,0.0002,차이 불확실
ll_joint,-0.0037,-0.0055,-0.0018,좋아짐
nll_coord,-0.0000,-0.0004,0.0003,차이 불확실
dist,0.0036,-0.0020,0.0091,차이 불확실


model_09: 보여 준 구종 수 평균 2.39, 포함률 0.905
model_10: 보여 준 구종 수 평균 2.43, 포함률 0.909


### 🥇1등🥇
#### 2020 구종 log loss 1.0561 -> 1.0524 (−0.0037), 부트스트랩: 구종·동시 좋아짐, 24칸·좌표 NLL·거리 차이 불확실, 나빠진 점수 없음
- 2019도 −0.0047 (w를 2019로 고른 몫 포함, 정직한 교차 크기는 −0.0042)
- 부트스트랩 (2020, model_10 − model_09): 구종 −0.0037 (−0.0055 ~ −0.0018), 동시 −0.0037 / 24칸 −0.0001, 좌표 NLL 0.0000, 거리 +0.004cm (모두 0 포함)
- 누적 80% 보여 주기: 평균 2.39 -> 2.43개, 포함률 90.5% -> 90.9% -> 개수·포함률이 같이 늘어 날카로움은 조금 손해 (확률이 퍼짐), 같은 개수 비교(top2·top3)는 좋아짐
- 주의: macro F1 2019 −0.031, 2020 −0.014 (1등이 직구로 몰림, 위 진단) -> 확률(log loss)은 좋아지지만 "1등 하나만 보여 주기"로 쓰면 드문 구종을 덜 맞힘
- log loss는 좋아지지만 1등 정확도(2019 top1 −0.25%p)·macro F1이 떨어짐: 크게 틀리던 공은 줄고, 아슬아슬하던 공의 1등이 직구로 바뀜
- **적용 안 함 (보류, 사용자 결정 2026-11-10)**: 다음 비교 출발점은 model_09 그대로 / 나중에 다시 쓸 때 이 노트북·`results/model_10/`·`results/all_pitch_01/`을 그대로 가져옴


## 6. 저장

- `results/model_10/` -> 채택하면 `compare.ipynb` `ADOPTED`에 추가

In [17]:
fc.save_results(MODEL, pp, {"summary": summary})

저장: results\model_10 ['per_pitch.parquet', 'summary.csv']
